# Name That LLM: TF-IDF de caracteres e palavras com regressão logística

O texto de cada modelo carrega marcas de estilo: pontuação, conectivos, aberturas e fechamentos. Este notebook combina TF-IDF de n-gramas de caracteres e de palavras numa regressão logística com pesos balanceados, porque a métrica é a balanced accuracy.

A validação é por prompt (GroupKFold), porque o teste só tem prompts que não aparecem no treino. Os experimentos e o código com testes estão em github.com/juan-barross/name-that-llm.


In [ ]:
from pathlib import Path

Path("ntl").mkdir(exist_ok=True)

In [ ]:
%%writefile ntl/__init__.py


In [ ]:
%%writefile ntl/labels.py
LABELS = (
    "gpt-5.5",
    "gpt-5.6-luna",
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-6-astra",
    "gemini-3.1-pro-preview",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    "claude-fable-5",
    "claude-opus-4-6",
    "claude-opus-4-8",
    "claude-opus-5",
    "claude-sonnet-5",
    "qwen3.8-max",
    "glm-5.3",
    "glm-5.3-flash",
    "kimi-k3",
)


In [ ]:
%%writefile ntl/data.py
from pathlib import Path

import pandas as pd

KAGGLE_DIR = Path("/kaggle/input/competitions/name-that-llm")
LOCAL_DIR = Path("data")


def data_dir() -> Path:
    return KAGGLE_DIR if KAGGLE_DIR.exists() else LOCAL_DIR


def load(name: str, root: Path | None = None) -> pd.DataFrame:
    return pd.read_csv((root or data_dir()) / f"{name}.csv")


In [ ]:
%%writefile ntl/features.py
import re

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

MARKS = {
    "em_dash": "—",
    "en_dash": "–",
    "semicolon": ";",
    "colon": ":",
    "parenthesis": r"\(",
    "curly_quote": "[“”‘’]",
    "straight_quote": "[\"']",
    "ellipsis": r"…|\.\.\.",
    "asterisk": r"\*",
    "hash": "#",
    "digit": r"\d",
    "exclamation": "!",
    "question": r"\?",
    "comma": ",",
    "newline": "\n",
}
PATTERNS = {name: re.compile(pattern) for name, pattern in MARKS.items()}
SENTENCE = re.compile(r"(?<=[.!?])\s+")


def text_style(text: str) -> list[float]:
    words = text.split()
    sentences = [len(s.split()) for s in SENTENCE.split(text) if s.strip()]
    chars = max(len(text), 1)
    rates = [len(pattern.findall(text)) / chars for pattern in PATTERNS.values()]
    return [
        *rates,
        np.log1p(len(words)),
        np.mean([len(w) for w in words]) if words else 0.0,
        np.mean(sentences) if sentences else 0.0,
        np.std(sentences) if sentences else 0.0,
        sum(c.isupper() for c in text) / chars,
    ]


def style_matrix(texts) -> np.ndarray:
    return np.array([text_style(text) for text in texts], dtype=np.float64)


def char_tfidf() -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, lowercase=False
    )


def word_tfidf() -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=r"\S+"
    )


def style() -> object:
    return make_pipeline(FunctionTransformer(style_matrix), StandardScaler())


def vectorizer(*parts: str) -> FeatureUnion:
    builders = {"char": char_tfidf, "word": word_tfidf, "style": style}
    return FeatureUnion([(part, builders[part]()) for part in parts])


In [ ]:
%%writefile ntl/models.py
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from ntl.features import vectorizer

EXPERIMENTS = {
    "char_logreg": (("char",), 10.0),
    "char_word_logreg": (("char", "word"), 10.0),
    "char_word_style_logreg": (("char", "word", "style"), 10.0),
}


def build(name: str) -> Pipeline:
    parts, strength = EXPERIMENTS[name]
    classifier = LogisticRegression(C=strength, class_weight="balanced", max_iter=3000)
    return Pipeline([("features", vectorizer(*parts)), ("classifier", classifier)])


In [ ]:
%%writefile ntl/validation.py
from collections.abc import Callable
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline

from ntl.labels import LABELS

SEED = 2027
N_SPLITS = 5


@dataclass(frozen=True)
class CrossValidation:
    probabilities: np.ndarray
    fold_scores: list[float]

    @property
    def score(self) -> float:
        return float(np.mean(self.fold_scores))


def folds(groups: pd.Series, n_splits: int = N_SPLITS) -> list[tuple[np.ndarray, np.ndarray]]:
    splitter = GroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    return list(splitter.split(groups, groups=groups))


def aligned_probabilities(model: Pipeline, texts: pd.Series) -> np.ndarray:
    columns = [list(model.classes_).index(label) for label in LABELS]
    return model.predict_proba(texts)[:, columns]


def cross_validate(build: Callable[[], Pipeline], train: pd.DataFrame) -> CrossValidation:
    probabilities = np.zeros((len(train), len(LABELS)))
    scores = []
    for fit_index, valid_index in folds(train["prompt_id"]):
        model = build().fit(train["text"].iloc[fit_index], train["model"].iloc[fit_index])
        probabilities[valid_index] = aligned_probabilities(model, train["text"].iloc[valid_index])
        predicted = np.array(LABELS)[probabilities[valid_index].argmax(axis=1)]
        scores.append(balanced_accuracy_score(train["model"].iloc[valid_index], predicted))
    return CrossValidation(probabilities, scores)


In [ ]:
%%writefile ntl/predict.py
import numpy as np
import pandas as pd

from ntl.labels import LABELS
from ntl.models import build
from ntl.validation import aligned_probabilities


def fit_predict(experiment: str, train: pd.DataFrame, test: pd.DataFrame) -> np.ndarray:
    model = build(experiment).fit(train["text"], train["model"])
    return np.array(LABELS)[aligned_probabilities(model, test["text"]).argmax(axis=1)]


In [ ]:
%%writefile ntl/submission.py
from pathlib import Path

import pandas as pd

from ntl.labels import LABELS

COLUMNS = ["id", "model"]


def validate(submission: pd.DataFrame) -> None:
    if list(submission.columns) != COLUMNS:
        raise ValueError(f"colunas devem ser {COLUMNS}, vieram {list(submission.columns)}")
    if submission.isna().any().any():
        raise ValueError("submissão com valor nulo")
    if submission["id"].duplicated().any():
        raise ValueError("submissão com id repetido")
    unknown = set(submission["model"]) - set(LABELS)
    if unknown:
        raise ValueError(f"rótulos fora da lista: {sorted(unknown)}")


def write(ids: pd.Series, predictions, path: Path = Path("submission.parquet")) -> pd.DataFrame:
    submission = pd.DataFrame({"id": ids.to_numpy(), "model": list(predictions)})
    validate(submission)
    submission.to_parquet(path, index=False)
    return submission


In [ ]:
from ntl.data import load
from ntl.predict import fit_predict
from ntl.submission import write

EXPERIMENT = "char_word_logreg"

train, test = load("train"), load("test")
submission = write(test["id"], fit_predict(EXPERIMENT, train, test))
submission["model"].value_counts()